# CytoVI (protein-only) benchmark — Patient 1 (03H096 / PB2)

**Role in the paper:** Cross-modal benchmark (Extended Data Fig. 7). It asks how much
of the trimodal structure — in particular the Root/CSC cluster — can be
recovered from the antibody-capture panel alone.

**What this notebook does**
1. Loads the raw MuData and keeps the barcodes retained by MultiVI
2. Runs protein (ADT) quality control and removes the isotype controls
3. Trains **CytoVI** on the antibody matrix
4. Builds neighbours, Leiden clusters and a ForceAtlas2 layout
5. Writes `PB2_CytoVI_adata.h5ad`, consumed by `04_Single_modality_checks/LSC_recovery_Patient1_03H096.ipynb`

**Objects**
- **Reads:** `DATA_DIR / "Teaseq_PB2.h5mu"` and
  `DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/Teaseq_Multi_VI_PB2_Cleaned.h5ad"`
- **Creates:** `DATA_DIR / "04_Single_modality_checks/Protein/PB2_CytoVI_adata.h5ad"`
  and the trained model directory next to it

## Paths and settings

In [ ]:
from pathlib import Path

# Root of the companion data package. Point this at your local copy.
DATA_DIR = Path("PATH_TO_DATA")  # <-- set this to your local data root
OUT_DIR = DATA_DIR / "outputs/CytoVI_PB2"     # figures and tables written by this notebook
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os

import anndata as ad
import matplotlib.pyplot as plt
import mudata as md
import muon
import numpy as np
import pandas as pd
import scanpy as sc
import scvi

## Load the raw MuData and the cleaned MultiVI reference

In [ ]:
### Read the H5 file ###

adataPB2 = muon.read(DATA_DIR / "Teaseq_PB2.h5mu")
adataPB2.var_names_make_unique()

In [ ]:
# Cleaned MultiVI object: source of the barcode list and cluster labels.
adataMultiPB2 = ad.read_h5ad(
    DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/Teaseq_Multi_VI_PB2_Cleaned.h5ad"
)

In [ ]:
# Normalise the antibody names to the ADT.anti.hu.<clone> convention.
adataPB2.mod['protein'].var.index = 'ADT.anti.hu.' + adataPB2.mod['protein'].var.index.str.split('-').str[0]

In [ ]:
### Annotation of all the samples ###

adataPB2.obs.index = [name + '_PB2' for name in adataPB2.obs_names]

In [ ]:
adata = adataPB2

## Split modalities and restrict to the QC-passing barcodes

In [ ]:
### Extracting RNA data only ###

rna_adata = adata.mod['rna']
prot_adata = adata.mod['protein']
rna_adata.obs.index = adata.obs.index
prot_adata.obs.index = adata.obs.index

In [ ]:
# Barcodes retained by MultiVI; every modality is subset to this list.
cell_ids_list = adataMultiPB2.obs_names.tolist()
print(len(cell_ids_list), 'cells;', cell_ids_list[:3])

In [ ]:
# Filter the MuData object
prot_adata = prot_adata[prot_adata.obs.index.isin(cell_ids_list)]
prot_adata

## Protein (ADT) quality control

In [ ]:
### Top ranking expressed gene ###

sc.pl.highest_expr_genes(prot_adata, n_top=20, )

In [ ]:
### Three plot summary BEFORE filtering ###

prot_adata.var['mt'] = prot_adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(prot_adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

prot_adata.var_names_make_unique()

sc.pl.violin(prot_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

prot_adata

In [ ]:
### Summary of genes and counts BEFORE filtering ###

sc.pl.scatter(prot_adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(prot_adata, x='total_counts', y='n_genes_by_counts')

## Remove the isotype controls

In [ ]:
# Filter out variables that contain 'IgG' in their names
prot_adata = prot_adata[:, ~prot_adata.var.index.str.contains('Rat|Mouse|Hamster')]
prot_adata

## Train CytoVI, embed and save

In [ ]:
import random

import torch
from scvi.external import cytovi

PATIENT_NAME = "PB2"  # sample processed by this run

# CytoVI is trained on the antibody-capture matrix alone.
adata = prot_adata.copy()

# Model and embedding are written next to the other single-modality results.
pipeline_dir = DATA_DIR / "04_Single_modality_checks/Protein"
pipeline_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {pipeline_dir}")

os.environ["SCIPY_ARRAY_API"] = "1"

sc.set_figure_params(figsize=(4, 4))

scvi.settings.seed = 0
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("Last run with scvi-tools version:", scvi.__version__)
print(f"Running patient: {PATIENT_NAME}")

# RUN CYTOVI
cytovi.CYTOVI.setup_anndata(adata)

model = cytovi.CYTOVI(adata)

model.train(n_epochs_kl_warmup=50)

# SAVE MODEL
model_path = pipeline_dir / f"{PATIENT_NAME}_CytoVI_Model"

model.save(model_path, overwrite=True)

print(f"Model saved successfully to: {model_path}")

# TRAINING CURVE
plt.plot(model.history["elbo_train"], label="Train")
plt.plot(model.history["elbo_validation"], label="Validation")
plt.xlabel("Epochs")
plt.ylabel("ELBO")
plt.legend()
plt.title(f"{PATIENT_NAME} Training vs Validation ELBO")
plt.show()

adata.obsm["X_CytoVI"] = model.get_latent_representation()

# NEIGHBORS / GRAPH
sc.pp.neighbors(
    adata,
    use_rep="X_CytoVI",
    transformer="pynndescent"
)

sc.tl.draw_graph(adata, layout="fa")

# CLUSTERING
sc.tl.leiden(
    adata,
    resolution=0.4,
    key_added="leiden_CytoVI",
    flavor="igraph"
)

# PLOT
muon.pl.embedding(
    adata,
    basis="X_draw_graph_fa",
    color=["leiden_CytoVI"],
    frameon=False,
    ncols=1,
)

# SAVE ANNDATA
adata_path = pipeline_dir / f"{PATIENT_NAME}_CytoVI_adata.h5ad"

adata.write_h5ad(adata_path)

print(f"AnnData saved successfully to: {adata_path}")

# DONE
print(f"{PATIENT_NAME} pipeline completed successfully.")